In [ ]:
import mne
import torch
import numpy 
from torch import nn
from moabb.datasets import BNCI2014_001
from braindecode.models import ShallowFBCSPNet
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
# used model and dataset
dataset = BNCI2014_001()
data = dataset.get_data()
model = ShallowFBCSPNet(n_chans=22, n_classes=4, input_window_samples=576, final_conv_length='auto')

In [ ]:
# defined loss function and optimizer
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params = model.parameters(), 
                            lr=0.01)

In [ ]:
# WITHIN-SUBJECT division

subjects = list(data.keys())

def epochs_from_session(session_dict, tmin=0, tmax=4.0):

    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# dicionário: subject_id -> {"train": epochs_sessao0, "test": epochs_sessao1}
within_subject_tensors = {}

for subj in subjects:
    session_keys = list(data[subj].keys())

    train_epochs = epochs_from_session(data[subj][session_keys[0]])
    test_epochs = epochs_from_session(data[subj][session_keys[1]])

    X_tr = train_epochs.get_data() * 1e6
    y_tr = train_epochs.events[:, -1]
    X_te = test_epochs.get_data() * 1e6
    y_te = test_epochs.events[:, -1]

    min_lbl = y_tr.min()

    within_subject_tensors[subj] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_tr - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_te, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_te - min_lbl, dtype=torch.long),
    }

for subj, tensors in within_subject_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

epochs = epochs_from_session(within_subject_tensors, 0, 4.0)
train_loss_values = [] # keep track of the loss values
test_loss_values = []
epoch_count = []

for epoch in range(epochs):
    model.train() 
    #forward pass
    y_pred = model(X_train_sub)

    #calculate the loss
    loss = loss_fn(y_pred, y_train_sub)

    #zero grad optimizer
    optimizer.zero_grad()

    #loss backward
    loss.backward()

    #step optimizer
    optimizer.step()

    #testing

    model.eval()

    with torch.inference_mode():
        test_pred = model(X_test_sub)

        test_loss = loss_fn(test_pred, y_test_sub)

    if epoch % 100 == 0:
        epoch_count.append(epoch)
        train_loss_values.append(loss.detach().numpy())
        test_loss_values.append(test_loss_values.detach().numpy())
        print(f"Epoch: {epoch} | Train loss: {loss} | Test loss: {test_loss}")

In [ ]:
# plot predictions

def plot_predictions(train_data=X_train_sub,
                      train_labels=y_train_sub,
                      test_data=X_test_sub,
                      test_labels=y_test_sub,
                      predictions=None):
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=train_data.flatten(),
        y=train_labels.flatten(),
        mode="markers",
        marker=dict(color="blue", size=4),
        name="Training data"
    ))


    fig.add_trace(go.Scatter(
        x=test_data.flatten(),
        y=test_labels.flatten(),
        mode="markers",
        marker=dict(color="green", size=4),
        name="Testing data"
    ))

    if predictions is not None:
        fig.add_trace(go.Scatter(
            x=test_data.flatten(),
            y=predictions.flatten(),
            mode="markers",
            marker=dict(color="red", size=4),
            name="Predictions"
        ))

    fig.update_layout(
        width=800,
        height=600,
        title="Predictions vs Data",
        xaxis_title="X",
        yaxis_title="y",
        legend=dict(x=0.02, y=0.98)
    )

    fig.show()
 

In [ ]:
# plot the loss curve
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=epoch_count.flatten(),
    y=train_loss_values.flatten(),
    mode="markers",
    marker=dict(color="blue", size=4),
    name="Train Loss"
    ))

fig.add_trace(go.Scatter(
    x=epoch_count.flatten(),
    y=test_loss_values.flatten(),
    mode="markers",
    marker=dict(color="blue", size=4),
    name="Test Loss"
    ))

fig.update_layout(
    width=800,
    height=600,
    title="Training and test loss curve",
    xaxis_title="Epochs",
    yaxis_title="Loss",
    legend=dict(x=0.02, y=0.98)
    )

fig.show()
 